In [1]:
import pandas as pd
df = pd.read_csv("egypt_real_estate_listings.csv", encoding="utf-8")
print(df.shape)
df.head()

(19924, 11)


,url,price,description,location,type,size,bedrooms,bathrooms,available_from,payment_method,down_payment
0,https://www.propertyfinder.eg/en/plp/buy/chale...,"8,000,000",OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...,"Swan Lake Gouna, Al Gouna, Hurghada, Red Sea",Chalet,732 sqft / 68 sqm,1+ Maid,1,31 Aug 2025,Cash,"1,200,000 EGP"
1,https://www.propertyfinder.eg/en/plp/buy/villa...,"25,000,000","For sale, a villa with immediate delivery in C...","Karmell, New Zayed City, Sheikh Zayed City, Giza",Villa,"2,368 sqft / 220 sqm",4,4,2 Sep 2025,Cash,"2,100,000 EGP"
2,https://www.propertyfinder.eg/en/plp/buy/chale...,"15,135,000","With a down payment of EGP 1,513,000, a fully ...","Azha North, Ras Al Hekma, North Coast",Chalet,"1,270 sqft / 118 sqm",2,2,19 Aug 2025,Cash,"1,513,000 EGP"
3,https://www.propertyfinder.eg/en/plp/buy/apart...,"12,652,000",Own an apartment in New Cairo with a minimal d...,"Taj City, 5th Settlement Compounds, The 5th Se...",Apartment,"1,787 sqft / 166 sqm",3,2,26 Aug 2025,Installments,"1,260,000 EGP"
4,https://www.propertyfinder.eg/en/plp/buy/villa...,"45,250,000",Project: Granville\nLocation: Fifth Settlement...,"Granville, New Capital City, Cairo",Villa,"4,306 sqft / 400 sqm",7,7,2 Sep 2025,Cash,"2,262,500 EGP"


In [2]:
# Data quality check on the original dataset
df.info()
print("-" * 40)
print("Missing values per column:")
print(df.isnull().sum())
print("-" * 40)
print("Duplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 19924 entries, 0 to 19923
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   url             19924 non-null  str  
 1   price           19385 non-null  str  
 2   description     19846 non-null  str  
 3   location        19833 non-null  str  
 4   type            19847 non-null  str  
 5   size            19847 non-null  str  
 6   bedrooms        19780 non-null  str  
 7   bathrooms       19784 non-null  str  
 8   available_from  19261 non-null  str  
 9   payment_method  19383 non-null  str  
 10  down_payment    5445 non-null   str  
dtypes: str(11)
memory usage: 1.7 MB
----------------------------------------
Missing values per column:
url                   0
price               539
description          78
location             91
type                 77
size                 77
bedrooms            144
bathrooms           140
available_from      663
payment_method      54

In [3]:
# 1) Work on a copy so the original data stays untouched
clean = df.copy()

# 2) Price: strip everything except digits, then convert to a number
clean["price"] = (
    clean["price"].str.replace(r"[^\d]", "", regex=True)
)
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")

# 3) Size: extract the number that comes before "sqm"
clean["size_sqm"] = pd.to_numeric(
    clean["size"].str.extract(r"([\d,]+)\s*sqm")[0].str.replace(",", ""),
    errors="coerce"
)

# 4) Check the result
print(clean[["price", "size", "size_sqm"]].head(10))
print(clean[["price", "size_sqm"]].describe())

        price                  size  size_sqm
0   8000000.0     732 sqft / 68 sqm      68.0
1  25000000.0  2,368 sqft / 220 sqm     220.0
2  15135000.0  1,270 sqft / 118 sqm     118.0
3  12652000.0  1,787 sqft / 166 sqm     166.0
4  45250000.0  4,306 sqft / 400 sqm     400.0
5  19760000.0  1,356 sqft / 126 sqm     126.0
6   9000000.0  2,820 sqft / 262 sqm     262.0
7   2770000.0  3,983 sqft / 370 sqm     370.0
8  13200000.0  2,368 sqft / 220 sqm     220.0
9  22300000.0  2,099 sqft / 195 sqm     195.0
              price      size_sqm
count  1.938500e+04  1.984700e+04
mean   1.641531e+07  1.034563e+03
std    2.369791e+07  1.122570e+05
min    1.869000e+05  1.000000e+00
25%    6.000000e+06  1.270000e+02
50%    1.034555e+07  1.700000e+02
75%    1.816900e+07  2.410000e+02
max    8.400000e+08  1.581169e+07


In [4]:
# Check the distribution of property sizes instead of guessing thresholds
print(clean["size_sqm"].quantile([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]))
print("Listings larger than 3000 sqm:", (clean["size_sqm"] > 3000).sum())
print("Listings smaller than 20 sqm:", (clean["size_sqm"] < 20).sum())

0.001      30.0
0.010      50.0
0.050      73.0
0.500     170.0
0.950     480.0
0.990     922.7
0.999    2250.0
Name: size_sqm, dtype: float64
Listings larger than 3000 sqm: 12
Listings smaller than 20 sqm: 2


In [5]:
# 1) Drop rows missing the key columns (price, size, type, location)
before = len(clean)
clean = clean.dropna(subset=["price", "size_sqm", "type", "location"])

# 2) Down payment: missing means no down payment (cash deals), so fill with 0
clean["down_payment"] = clean["down_payment"].astype(str).str.replace(r"[^\d]", "", regex=True)
clean["down_payment"] = pd.to_numeric(clean["down_payment"], errors="coerce").fillna(0)

# 3) Remove impossible sizes (thresholds confirmed with quantile analysis)
clean = clean[(clean["size_sqm"] >= 20) & (clean["size_sqm"] <= 3000)]

# 4) Remove unrealistic prices (below 100k EGP)
clean = clean[clean["price"] >= 100_000]

# 5) Summary
print("Rows before cleaning:", before, "| after cleaning:", len(clean))
print(clean[["price", "size_sqm"]].describe())

Rows before cleaning: 19924 | after cleaning: 19377
              price      size_sqm
count  1.937700e+04  19377.000000
mean   1.636315e+07    210.839604
std    2.315664e+07    166.060073
min    1.869000e+05     20.000000
25%    6.000000e+06    126.000000
50%    1.034400e+07    170.000000
75%    1.815000e+07    239.000000
max    8.400000e+08   3000.000000


In [6]:
# Check the price distribution before deciding on any threshold
print(clean["price"].quantile([0.001, 0.01, 0.5, 0.99, 0.999]))

# Look at the 10 most expensive listings with their size and type
top = clean.nlargest(10, "price")[["price", "size_sqm", "type", "location"]]
print(top)

0.001    5.000000e+05
0.010    1.371520e+06
0.500    1.034400e+07
0.990    1.000000e+08
0.999    2.762400e+08
Name: price, dtype: float64
             price  size_sqm   type  \
14762  840000000.0    1100.0  Villa   
18560  453000000.0     610.0  Villa   
12051  400000000.0    1100.0  Villa   
13447  400000000.0    2000.0  Villa   
15066  400000000.0    1101.0  Villa   
9871   375000000.0    2801.0  Villa   
13068  375000000.0     696.0  Villa   
12370  374000000.0     696.0  Villa   
11523  360000000.0     700.0  Villa   
12594  350000000.0    1700.0  Villa   

                                                location  
14762            Marassi, Sidi Abdel Rahman, North Coast  
18560            Marassi, Sidi Abdel Rahman, North Coast  
12051  Katameya Heights, El Katameya Compounds, El Ka...  
13447  Katameya Dunes, El Katameya Compounds, El Kata...  
15066  Katameya Heights, El Katameya Compounds, El Ka...  
9871   Katameya Dunes, El Katameya Compounds, El Kata...  
13068            Ma

In [7]:
# Compare each villa's price per sqm to the median of its own compound
clean["price_per_sqm"] = clean["price"] / clean["size_sqm"]
clean["compound"] = clean["location"].str.split(",").str[0].str.strip()

villas = clean[clean["type"] == "Villa"].copy()
villas["compound_median"] = villas.groupby("compound")["price_per_sqm"].transform("median")
villas["ratio_to_median"] = villas["price_per_sqm"] / villas["compound_median"]

cols = ["price", "size_sqm", "price_per_sqm", "compound", "ratio_to_median", "url"]
print(villas.nlargest(10, "price")[cols])

             price  size_sqm  price_per_sqm          compound  \
14762  840000000.0    1100.0  763636.363636           Marassi   
18560  453000000.0     610.0  742622.950820           Marassi   
12051  400000000.0    1100.0  363636.363636  Katameya Heights   
13447  400000000.0    2000.0  200000.000000    Katameya Dunes   
15066  400000000.0    1101.0  363306.085377  Katameya Heights   
9871   375000000.0    2801.0  133880.756873    Katameya Dunes   
13068  375000000.0     696.0  538793.103448           Marassi   
12370  374000000.0     696.0  537356.321839           Marassi   
11523  360000000.0     700.0  514285.714286           Marassi   
12594  350000000.0    1700.0  205882.352941  Katameya Heights   

       ratio_to_median                                                url  
14762         5.031663  https://www.propertyfinder.eg/en/plp/buy/villa...  
18560         4.893204  https://www.propertyfinder.eg/en/plp/buy/villa...  
12051         1.575758  https://www.propertyfinder.eg/en

In [8]:
# 1) Region (last part of the location text); compound and price_per_sqm already exist
parts = clean["location"].str.split(",")
clean["area"] = parts.str[0].str.strip()
clean["region"] = parts.str[-1].str.strip()

# 2) Bedrooms and bathrooms: extract the first number (e.g. "1+ Maid" becomes 1)
clean["bedrooms_num"] = pd.to_numeric(clean["bedrooms"].str.extract(r"(\d+)")[0], errors="coerce")
clean["bathrooms_num"] = pd.to_numeric(clean["bathrooms"].str.extract(r"(\d+)")[0], errors="coerce")

# 3) Flag whether the bedrooms value mentions a maid's room
clean["has_maid_room"] = clean["bedrooms"].str.contains("Maid", case=False, na=False)

# 4) Quick check of the new columns
print(clean[["price_per_sqm", "compound", "region", "bedrooms", "bedrooms_num", "has_maid_room"]].head(10))
print(clean["region"].value_counts().head(10))

   price_per_sqm               compound       region bedrooms  bedrooms_num  \
0  117647.058824        Swan Lake Gouna      Red Sea  1+ Maid           1.0   
1  113636.363636                Karmell         Giza        4           4.0   
2  128262.711864             Azha North  North Coast        2           2.0   
3   76216.867470               Taj City        Cairo        3           3.0   
4  113125.000000              Granville        Cairo        7           7.0   
5  156825.396825               Marina 5  North Coast        3           3.0   
6   34351.145038     The Icon Residence        Cairo        4           4.0   
7    7486.486486          Nyoum October         Giza  5+ Maid           5.0   
8   60000.000000  La vista Ras El Hikma  North Coast        3           3.0   
9  114358.974359                   Azad        Cairo  3+ Maid           3.0   

   has_maid_room  
0           True  
1          False  
2          False  
3          False  
4          False  
5          False

In [9]:
# Keep only the columns needed for analysis, in a clean order
final_cols = [
    "url", "price", "size_sqm", "price_per_sqm", "type", "bedrooms_num",
    "bathrooms_num", "has_maid_room", "compound", "region", "available_from",
    "payment_method", "down_payment", "description",
]
final = clean[final_cols].copy()

# Convert the availability date text into a real date
final["available_from"] = pd.to_datetime(final["available_from"], errors="coerce")

# Save to CSV (utf-8-sig so Excel displays text correctly)
final.to_csv("egypt_real_estate_clean.csv", index=False, encoding="utf-8-sig")

print("Saved", len(final), "rows")
print(final.dtypes)

Saved 19377 rows
url                          str
price                    float64
size_sqm                 float64
price_per_sqm            float64
type                         str
bedrooms_num             float64
bathrooms_num            float64
has_maid_room               bool
compound                  object
region                    object
available_from    datetime64[us]
payment_method               str
down_payment             float64
description                  str
dtype: object
